# Paper: Scaling Laws and Compute-Optimal Training BEyond Fixed Training Durations

this paper figures out the empiral law of learning rate scheduler. By experimenting with different learnign rate across different settings, they find out the loss landscape correspond to the type of scheduler used.




Starting point

1. Cosine learning-rate scheduling is the standard choice in many LLM training recipes. The paper starts from the observation that cosine decay has remained the de facto default in large-model pretraining.

2. Key property of matched-length cosine:
To get the best perplexity at a given training length, the cosine schedule must be matched to that exact horizon. In other words, the total training length becomes part of the recipe.

![image.png](.\assets\figure1_lrsche.png)


Problem:

1. makes training more expensive for scaling experiments

2. complicates continuation of training 

3. Rewarming is not a clean fix




## Main alternative 

- keep the learning rate constant for most of training 

- apply a short colldown at the end


#### Experiment one 


![image.png](.\assets\fig2_LRSCHE.png)

- you do not need a full cosine decay over the whole run

- a late cooldown phase can recover almost the same final perplexity

- that means the important part may be the ending anneal, not the fact that cosine has been slowly decaying the whole time. 


![image.png](.\assets\fig3_LRSCHE.png)

Compare:

- linear cooldown
- $1 - \sqrt{\cdot}$ cooldown
- cosine as a reference

They find that the $1 - \sqrt{\cdot}$ cooldown consistently outperforms the linear cooldown, and for longer runs both cooldown variants can outperform the chosen untuned cosine baseline in that experiment.

The cooldown is defined during the decay phase as:

$$
f(n, N, N_{\text{decay}}) = 1 - \sqrt{\frac{n - (N - N_{\text{decay}})}{N_{\text{decay}}}}
$$

where:

- $N$ is the total number of training steps
- $N_{\text{decay}}$ is the number of cooldown steps
- $n$ is the current training step

Intuition:

- **Linear cooldown** decreases at a constant rate.
- **$1 - \sqrt{\cdot}$ cooldown** drops faster at the start of the cooldown, then flattens later.


![image.png](.\assets\fig5_LRSCHE.png)

### Figure 5: How long should cooldown be?

Figure 5 investigates cooldown length as a fraction of total training steps. Increasing the cooldown helps reduce perplexity at first, and cooldown begins to outperform cosine at around 10–20% of total steps. However, the benefit largely plateaus beyond that, so very long cooldowns do not provide much additional improvement.


![image.png](.\assets\fig6_LRSCHE.png)

### Figure 6: Short cooldown can still work for long runs

Figure 6 shows that for long training runs, the cooldown can be much shorter as a fraction of training while still matching cosine. In particular, a 5% cooldown nearly matches cosine in a long run, suggesting that what matters is not only the fraction of training spent cooling down, but whether the cooldown is long enough in absolute steps.

![image.png](.\assets\fig10_LRSCHE.png)


### Figure 10: SWA improves models during training

Figure 10 shows that stochastic weight averaging (SWA) improves performance along the training trajectory.

- On top of a constant learning rate, SWA gives better perplexity than the plain constant-LR run.
- This suggests SWA improves generalization and can partially simulate the effect of learning-rate decay.
- However, SWA still does not fully match the performance of an explicit cooldown.
- Therefore, SWA is a useful low-cost method for obtaining better intermediate checkpoints, while a cooldown such as $1 - \sqrt{\cdot}$ can still be applied later if lower final loss is desired.

![image.png](.\assets\fig12_LRSCHE.png)


### Figure 12: Cooldown and SWA scale reliably

Figure 12 shows that cooldown and SWA preserve nearly the same scaling behavior as cosine across model sizes. When comparing perplexity at the same token counts, the points lie close to the diagonal, meaning cooldown achieves almost the same loss as cosine. SWA also tracks cosine reasonably well, although it remains slightly worse than explicit cooldown.

### Figure 13: Similar loss with less compute

Because cooldown can be applied retrospectively from reusable constant-LR runs, scaling-law experiments require fewer separate trainings. Figure 13 shows that this substantially reduces both FLOPs and GPU hours. In the paper’s Chinchilla-style estimate, the total compute drops from roughly $5.59 \times 10^{23}$ FLOPs to $2.36 \times 10^{23}$ FLOPs.